In [ ]:
import ndlib.models.ModelConfig as mc
import ndlib.models.epidemics as ep
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

### Function to Create a Network

In [ ]:
def create_network(network_type, num_nodes):
    if network_type == 'Erdos-Reyni':
        return nx.erdos_renyi_graph(num_nodes, 0.1)
    elif network_type == 'Barabasi-Albert':
        return nx.barabasi_albert_graph(num_nodes, 5)
    elif network_type == 'Watts-Strogatz':
        return nx.watts_strogatz_graph(num_nodes, 6, 0.1)
    else:
        raise ValueError("Invalid network type. Choose from 'Erdos-Reyni', 'Barabasi-Albert', or 'Watts-Strogatz'.")

### Function to Run the Simulation on an SIR Model 

In [ ]:
# Function to run SIR simulation
def run_sir_simulation(network, beta, gamma, initial_infected_fraction, iterations=100):
    # Configuring the model
    model = ep.SIRModel(network)
    config = mc.Configuration()

    #Add model Parameters 
    config.add_model_parameter('beta', beta) 
    config.add_model_parameter('gamma', gamma)  
    config.add_model_parameter('percentage_infected', initial_infected_fraction)  # Initial infected population 
    model.set_initial_status(config)

    trends = model.iteration_bunch(iterations)

    susceptible = [trend['node_count'][0] for trend in trends]
    infected = [trend['node_count'][1] for trend in trends]
    recovered = [trend['node_count'][2] for trend in trends]

    return susceptible, infected, recovered

### Function to Plot SIR Results

In [ ]:
# Function to plot SIR curves for all combinations of beta and gamma
def plot_sir_results(results):
    num_plots = len(results)
    cols = 3  # Number of columns in the plot grid
    rows = (num_plots // cols) + (num_plots % cols > 0)  # Calculate number of rows needed
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = axes.flatten()  # Flatten the axes array to easily iterate through it

    for idx, ((beta, gamma), (susceptible, infected, recovered)) in enumerate(results.items()):
        ax = axes[idx]
        ax.plot(susceptible, label='Susceptible', color='blue')
        ax.plot(infected, label='Infected', color='red')
        ax.plot(recovered, label='Recovered', color='green')

        ax.set_title(f"Beta: {beta}, Gamma: {gamma}")
        ax.set_xlabel('Iterations')
        ax.set_ylabel('Population')
        ax.legend()

    # Hide any unused subplots
    for idx in range(len(results), len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.show()

### Function to Run Experiments with Varying Parameters

In [ ]:
# Function to run experiments with varying parameters
def experiment_with_varying_parameters(beta_values, gamma_values, network_type='Erdos-Reyni', num_nodes=100, initial_infected_fraction=0.01, iterations=100):
    results = {}

    # Create the network
    network = create_network(network_type, num_nodes)

    # Run SIR simulations for all combinations of beta and gamma
    for beta in beta_values:
        for gamma in gamma_values:
            susceptible, infected, recovered = run_sir_simulation(network, beta, gamma, initial_infected_fraction, iterations)
            results[(beta, gamma)] = (susceptible, infected, recovered)

            plt.plot(infected, label=f'Beta: {beta}, Gamma: {gamma}')
    
    plt.title(f'SIR Model on {network_type.capitalize()} Network')
    plt.xlabel('Iterations')
    plt.ylabel('Infected Count')
    plt.legend()
    plt.show()

    return results

### Running the Experiment on Different Parameters 

In [ ]:
beta_values = np.linspace(0.1, 0.5, 2)  
gamma_values = np.linspace(0.1, 0.5, 2)  


results = experiment_with_varying_parameters(beta_values, gamma_values, network_type='Barabasi-Albert', num_nodes=200, iterations=100)
plot_sir_results(results)